# Lesson 02 — Logistic Regression

Lesson 01 predicted a continuous number. Now we predict a **class**: spam or not spam,
malignant or benign, click or no click. This is **binary classification**, and the labels
are $y \in \{0, 1\}$.

The machinery is the same as before. We define a model, define a cost function, derive
its gradient, and run gradient descent. Three of those four steps change, and the fourth
one, gradient descent, does not change at all.

What we build here:

1. the **sigmoid function**, which turns any real number into a probability,
2. the model $f_{w,b}(x) = \sigma(w \cdot x + b)$, interpreted as $P(y = 1 \mid x)$,
3. the **binary cross-entropy** cost, and why squared error is the wrong choice,
4. the gradient, which comes out with the *same form* as linear regression,
5. **classification metrics**: accuracy hides things that precision and recall reveal.

The reference implementation is in `logistic_regression.py`, tested by
`test_logistic_regression.py`.

## Notation

| symbol | meaning |
|---|---|
| $m, n$ | number of training examples, number of features |
| $X$ | design matrix of shape $(m, n)$, one row per example |
| $y$ | labels of shape $(m,)$, each entry either $0$ or $1$ |
| $z = w \cdot x + b$ | the **logit**, or linear score, an unbounded real number |
| $\sigma(z)$ | the sigmoid, mapping the logit into $(0, 1)$ |
| $f_{w,b}(x)$ | the model output, a **probability**, not a class |

Class $1$ is called the **positive class** and class $0$ the **negative class**. Those
names carry no value judgement. The positive class is simply the one you are detecting.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from logistic_regression import (
    sigmoid, predict_proba, compute_cost, compute_gradient, gradient_descent,
    LogisticRegression, accuracy, confusion_matrix, precision_recall_f1,
)

rng = np.random.default_rng(0)
plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True

## 1. Why not just use linear regression?

The obvious idea is to fit a straight line to labels that happen to be $0$ and $1$, then
predict class $1$ whenever the line sits above $0.5$. It even works on tidy data. It
breaks for two reasons.

**Reason one: the predictions are not probabilities.** A straight line is unbounded, so
it happily outputs $-0.4$ or $1.7$. There is no sensible reading of "this email is
$170\%$ spam".

**Reason two, the serious one: the fit is dragged around by points that are already
correct.** Squared error charges a penalty for a confident correct prediction, because
predicting $2.3$ when the label is $1$ costs $(2.3 - 1)^2$. The line tilts to reduce that
meaningless penalty, and the decision boundary moves as a side effect.

In [ ]:
import sys
sys.path.insert(0, "../01-linear-regression")
from linear_regression import normal_equation

x_clean = np.array([1.0, 2.0, 3.0, 4.0, 5.0, 6.0])
y_clean = np.array([0.0, 0.0, 0.0, 1.0, 1.0, 1.0])

# one extra example of class 1, sitting far to the right and already classified correctly
x_extra = np.append(x_clean, 20.0)
y_extra = np.append(y_clean, 1.0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (xd, yd, title, xmax) in zip(axes, [
        (x_clean, y_clean, "tidy data: the boundary lands correctly", 7),
        (x_extra, y_extra, "one far away point of class 1 is added", 21)]):
    w_lin, b_lin = normal_equation(xd.reshape(-1, 1), yd)
    boundary = (0.5 - b_lin) / w_lin[0]

    grid = np.linspace(0, xmax, 100)
    ax.scatter(xd[yd == 0], yd[yd == 0], c="tab:blue", s=60, label="class 0")
    ax.scatter(xd[yd == 1], yd[yd == 1], c="tab:red", s=60, label="class 1")
    ax.plot(grid, w_lin[0] * grid + b_lin, "k-", label="linear regression fit")
    ax.axhline(0.5, color="gray", ls=":", lw=1)
    ax.axvline(boundary, color="g", ls="--", label=f"boundary at x = {boundary:.2f}")
    ax.set_xlabel("$x$"); ax.set_ylabel("$y$"); ax.set_title(title); ax.legend(fontsize=8)
axes[0].set_ylabel("$y$")
plt.tight_layout(); plt.show()

w_e, b_e = normal_equation(x_extra.reshape(-1, 1), y_extra)
print(f"the example at x = 4 has label 1, and the model now scores it "
      f"{w_e[0] * 4 + b_e:.3f}, which is below 0.5, so it is misclassified")

The added point at $x = 20$ was already on the correct side by a wide margin. It taught
the model nothing new, yet it pushed the boundary from $3.5$ to $4.31$ and broke a
training example that used to be right.

Logistic regression fixes both problems at once. It squashes the output into $(0, 1)$ so
it reads as a probability, and it uses a cost function that stops charging you for
predictions that are already confidently correct.

## 2. The sigmoid function

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

It takes any real number and returns a value in $(0, 1)$, never reaching either endpoint.
Properties worth memorising:

| property | statement |
|---|---|
| midpoint | $\sigma(0) = 0.5$ |
| symmetry | $\sigma(-z) = 1 - \sigma(z)$ |
| limits | $\sigma(z) \to 1$ as $z \to \infty$, $\sigma(z) \to 0$ as $z \to -\infty$ |
| derivative | $\sigma'(z) = \sigma(z)\big(1 - \sigma(z)\big)$ |

That derivative identity is unusually convenient and does most of the work in section 6.
Derive it once by hand, using the quotient rule on $(1 + e^{-z})^{-1}$, and you will
recognise it everywhere in neural networks.

In [ ]:
z = np.linspace(-8, 8, 400)
s = sigmoid(z)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(z, s, lw=2)
axes[0].axhline(0.5, color="gray", ls=":"); axes[0].axvline(0, color="gray", ls=":")
axes[0].axhline(1.0, color="r", ls="--", lw=0.8); axes[0].axhline(0.0, color="r", ls="--", lw=0.8)
axes[0].set_xlabel("$z$"); axes[0].set_ylabel(r"$\sigma(z)$")
axes[0].set_title("The sigmoid squashes into (0, 1)")

axes[1].plot(z, s * (1 - s), lw=2, color="tab:orange")
axes[1].set_xlabel("$z$"); axes[1].set_ylabel(r"$\sigma'(z)$")
axes[1].set_title(r"Its derivative $\sigma(1-\sigma)$, largest at $z=0$")
plt.tight_layout(); plt.show()

print(f"sigmoid(0)   = {sigmoid(0.0):.4f}")
print(f"sigmoid(-2)  = {sigmoid(-2.0):.4f}   1 - sigmoid(2) = {1 - sigmoid(2.0):.4f}")
print(f"max of the derivative is {(s * (1 - s)).max():.4f} at z = 0")

### Computing the sigmoid without overflowing

The textbook formula $1/(1 + e^{-z})$ overflows when $z$ is a large negative number,
because $e^{-z}$ becomes larger than a float64 can hold. The fix is to use the algebraic
identity $\sigma(z) = e^{z}/(1 + e^{z})$ for negative $z$, so the exponential argument is
always negative and the result stays small.

```python
positive = z >= 0
out[positive]  = 1 / (1 + exp(-z[positive]))    # exponent is negative here
out[~positive] = exp(z[~positive]) / (1 + exp(z[~positive]))   # and negative here too
```

`out` is an output array of the same shape as `z`, and the two boolean masks select
complementary sets of its entries, so every entry is filled exactly once.

In [ ]:
extreme = np.array([-1000.0, -50.0, 0.0, 50.0, 1000.0])
print("our sigmoid :", sigmoid(extreme))

with np.errstate(over="ignore"):
    naive = 1.0 / (1.0 + np.exp(-extreme))
print("naive formula:", naive, "  (numpy raises an overflow warning, suppressed here)")

The naive version happens to return the right answer here, since dividing by infinity
gives zero. It raises an overflow warning on the way, and in longer calculations that
infinity propagates into `nan`. We will meet a case in section 5 where the naive route
genuinely fails.

## 3. The model

$$z = w \cdot x + b \qquad f_{w,b}(x) = \sigma(z) = \frac{1}{1 + e^{-(w \cdot x + b)}}$$

Read the output as a probability:

$$f_{w,b}(x) = P(y = 1 \mid x; w, b)$$

Given a house, an email, a tumour, the model returns its belief that the label is $1$.
Because the two classes are exhaustive, $P(y = 0 \mid x) = 1 - f_{w,b}(x)$.

### The decision boundary

To commit to a class, apply a **threshold**, conventionally $0.5$:

$$\hat{y} = 1 \iff f_{w,b}(x) \geq 0.5 \iff \sigma(z) \geq 0.5 \iff z \geq 0$$

The last step uses $\sigma(0) = 0.5$ together with the fact that the sigmoid increases
monotonically. So the **decision boundary is the set of points where $z = 0$**, that is

$$w \cdot x + b = 0$$

which is a straight line when $n = 2$, a plane when $n = 3$, and a hyperplane in general.
The sigmoid is nonlinear, but the boundary it produces is linear. Section 9 shows how to
bend it.

### Reading the weights as log-odds

Inverting the sigmoid gives $z$ back:

$$z = \log\frac{p}{1 - p} \qquad \text{where } p = f_{w,b}(x)$$

The quantity $p/(1-p)$ is the **odds**, and $z$ is the **log-odds**, also called the
logit. So logistic regression is a *linear model of the log-odds*. A weight $w_j$ has a
concrete meaning: increasing feature $j$ by one unit adds $w_j$ to the log-odds, which
multiplies the odds by $e^{w_j}$.

In [ ]:
m = 200
X = np.vstack([
    rng.normal([-1.5, -1.0], 1.0, size=(m // 2, 2)),
    rng.normal([1.5, 1.0], 1.0, size=(m // 2, 2)),
])
y = np.concatenate([np.zeros(m // 2), np.ones(m // 2)])
print("X.shape =", X.shape, "  y.shape =", y.shape, "  class counts:", np.bincount(y.astype(int)))


def plot_data(ax):
    ax.scatter(X[y == 0, 0], X[y == 0, 1], c="tab:blue", s=20, alpha=0.7, label="class 0")
    ax.scatter(X[y == 1, 0], X[y == 1, 1], c="tab:red", s=20, alpha=0.7, marker="^", label="class 1")
    ax.set_xlabel("$x_1$"); ax.set_ylabel("$x_2$")


def plot_boundary(ax, w, b, color="k", label="decision boundary"):
    # the boundary is w1*x1 + w2*x2 + b = 0, rearranged for x2
    x1 = np.array([X[:, 0].min() - 0.5, X[:, 0].max() + 0.5])
    ax.plot(x1, -(w[0] * x1 + b) / w[1], color=color, lw=2, label=label)


fig, ax = plt.subplots()
plot_data(ax)
plot_boundary(ax, np.array([1.0, 1.0]), 0.0, label="a guessed boundary")
ax.legend(); ax.set_title("Training data, and one guess at a boundary")
plt.show()

## 4. Why squared error is the wrong cost

Reusing lesson 01's cost with the new model gives

$$J_{\text{MSE}}(w, b) = \frac{1}{2m}\sum_{i=1}^{m}\left(\sigma(z^{(i)}) - y^{(i)}\right)^2$$

This is a legitimate function, and minimising it is not obviously wrong. It fails for two
separate reasons.

**Problem one: it is not convex.** In lesson 01 the squared error was a bowl with exactly
one minimum. Composing it with the sigmoid destroys that. The surface acquires flat
regions and local minima, so gradient descent can stall somewhere that is not the answer,
and where it stops depends on where it started.

In [ ]:
x_1d = np.array([1.0, 2.0, 3.0, 4.0, 5.0, 6.0])
y_1d = np.array([0.0, 0.0, 0.0, 1.0, 1.0, 1.0])


def cost_mse(x, y, w, b):
    return float(np.mean((sigmoid(w * x + b) - y) ** 2) / 2)


def cost_ce(x, y, w, b):
    z = w * x + b
    return float(np.mean(np.logaddexp(0.0, z) - y * z))


w_range = np.linspace(-6, 6, 601)
b_fixed = -3.5
J_mse = np.array([cost_mse(x_1d, y_1d, w, b_fixed) for w in w_range])
J_ce = np.array([cost_ce(x_1d, y_1d, w, b_fixed) for w in w_range])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, J, name in [(axes[0], J_mse, "squared error"), (axes[1], J_ce, "cross-entropy")]:
    curvature = np.diff(J, 2)
    ax.plot(w_range, J, lw=2)
    ax.fill_between(w_range[1:-1], J.min(), J.max(), where=curvature < 0,
                    color="tab:red", alpha=0.15, label="curving downward")
    ax.set_xlabel("$w$"); ax.set_ylabel("$J$")
    ax.set_title(f"{name}: {np.mean(curvature < 0):.0%} of this slice is not convex")
    ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

The shaded regions are where the curve bends downward, which a convex function never
does. Squared error is non-convex over most of this slice. Cross-entropy is convex
everywhere on it, and that holds in general, not just on this example.

**Problem two, which bites harder in practice: the gradient vanishes exactly when you
need it most.** Differentiating the squared error with respect to the logit $z$ brings
down a factor of $\sigma'(z)$ by the chain rule:

$$\frac{\partial}{\partial z}\;\frac{1}{2}\big(\sigma(z) - y\big)^2 = \big(\sigma(z) - y\big)\,\sigma'(z)$$

Now consider an example with label $y = 1$ that the model gets badly wrong, say
$z = -10$. The error $\sigma(z) - y$ is close to $-1$, which is as wrong as possible. But
$\sigma'(-10) \approx 4.5 \times 10^{-5}$, so their product is minuscule. The model is
maximally wrong and the gradient tells it to barely move.

Cross-entropy is built precisely so that this factor cancels.

In [ ]:
z_test = np.array([-10.0, -4.0, -1.0, 0.0, 1.0, 4.0, 10.0])
label = 1.0
s_test = sigmoid(z_test)
grad_mse = (s_test - label) * s_test * (1 - s_test)
grad_ce = s_test - label

print("An example whose true label is 1. How strongly does each cost push on z?")
print(f"{'z':>7}{'sigmoid(z)':>13}{'MSE gradient':>16}{'cross-entropy':>16}")
for zi, si, gm, gc in zip(z_test, s_test, grad_mse, grad_ce):
    print(f"{zi:>7.1f}{si:>13.6f}{gm:>16.6f}{gc:>16.6f}")
print("\nAt z = -10 the model is confidently wrong. Squared error pushes with magnitude")
print(f"{abs(grad_mse[0]):.2e}, while cross-entropy pushes with magnitude {abs(grad_ce[0]):.4f},")
print(f"a factor of {abs(grad_ce[0] / grad_mse[0]):.0f} difference.")

## 5. Binary cross-entropy

We want a loss that is small when the predicted probability agrees with the label and
grows without bound as the model becomes confidently wrong. For a single example:

$$
L\big(f_{w,b}(x), y\big) =
\begin{cases}
-\log\big(f_{w,b}(x)\big) & \text{if } y = 1 \\[4pt]
-\log\big(1 - f_{w,b}(x)\big) & \text{if } y = 0
\end{cases}
$$

Check the behaviour on the $y = 1$ branch. If the model predicts $0.99$, the loss is
$-\log(0.99) \approx 0.01$, which is nearly free. If it predicts $0.01$, the loss is
$-\log(0.01) \approx 4.6$, which is severe. As the prediction approaches $0$ the loss
grows to infinity. Being confidently wrong is punished without limit, which is exactly
the property squared error lacked.

The two branches combine into one expression, because $y$ is either $0$ or $1$ and so one
term always switches off:

$$L = -y\log\big(f_{w,b}(x)\big) - (1 - y)\log\big(1 - f_{w,b}(x)\big)$$

Averaging over the training set gives the cost:

$$\boxed{\;J(w,b) = \frac{1}{m}\sum_{i=1}^{m}\left[-y^{(i)}\log\big(f_{w,b}(x^{(i)})\big) - \big(1 - y^{(i)}\big)\log\big(1 - f_{w,b}(x^{(i)})\big)\right]\;}$$

Note there is no $\frac{1}{2}$ this time. It was only ever there to cancel the $2$ from
differentiating a square, and there is no square here.

### Where this comes from: maximum likelihood

The loss is not an arbitrary choice. Model each label as a Bernoulli draw with success
probability $f_{w,b}(x^{(i)})$. The probability of observing example $i$ is

$$P\big(y^{(i)} \mid x^{(i)}\big) = f^{\,y^{(i)}}\big(1 - f\big)^{1 - y^{(i)}}$$

which evaluates to $f$ when $y^{(i)} = 1$ and to $1 - f$ when $y^{(i)} = 0$. Assuming the
examples are independent, the likelihood of the whole dataset is the product of those
terms. Products of many small numbers underflow, and sums are easier to differentiate, so
take the logarithm. Maximising the log-likelihood is the same as minimising its negative,
and dividing by $m$ gives exactly the cost above. **Binary cross-entropy is the negative
log-likelihood of a Bernoulli model.**

In [ ]:
p = np.linspace(0.001, 0.999, 500)

plt.plot(p, -np.log(p), label="loss when the true label is 1")
plt.plot(p, -np.log(1 - p), label="loss when the true label is 0")
plt.xlabel("predicted probability $f_{w,b}(x)$"); plt.ylabel("loss")
plt.ylim(0, 5); plt.legend(); plt.title("Confident and wrong is punished without limit")
plt.show()

### Computing the cost without producing `nan`

The formula above is dangerous in floating point. Once $z$ passes roughly $37$, the
sigmoid rounds to exactly $1.0$, so $\log(1 - f)$ becomes $\log(0)$, which is $-\infty$,
and multiplying that by a zero coefficient gives `nan`. The cost of a single saturated
example then destroys the whole batch.

Rewrite the loss in terms of the logit $z$ rather than the probability. Substituting
$\log \sigma(z) = -\log(1 + e^{-z})$ and $\log(1 - \sigma(z)) = -z - \log(1 + e^{-z})$
and simplifying gives

$$\boxed{\;L = \log\big(1 + e^{z}\big) - y\,z\;}$$

and `np.logaddexp(0, z)` computes $\log(1 + e^z)$ without ever forming $e^z$ explicitly.
This identity is worth committing to memory. Every serious library implements the loss
this way, which is why they ask for logits rather than probabilities.

In [ ]:
def cost_naive(X, y, w, b):
    f = predict_proba(X, w, b)
    return float(np.mean(-y * np.log(f) - (1 - y) * np.log(1 - f)))


X_sat = np.array([[40.0], [-40.0]])     # logits of +40 and -40 once w = 1
y_sat = np.array([1.0, 0.0])            # both labels are correct and confident

with np.errstate(divide="ignore", invalid="ignore"):
    print("naive formula :", cost_naive(X_sat, y_sat, np.array([1.0]), 0.0))
print("stable formula:", compute_cost(X_sat, y_sat, np.array([1.0]), 0.0))
print("\nBoth predictions are correct, so the true cost is nearly zero.")
print("The naive version returns nan because log(1 - 1.0) is -inf and 0 * -inf is nan.")

## 6. The gradient

Here is the payoff for choosing cross-entropy. Differentiate the single-example loss with
respect to the logit $z$, using $\sigma'(z) = \sigma(1 - \sigma)$ and writing
$f = \sigma(z)$:

$$
\frac{\partial L}{\partial z}
= \left(-\frac{y}{f} + \frac{1-y}{1-f}\right)\underbrace{f(1-f)}_{\sigma'(z)}
= -y(1-f) + (1-y)f
= f - y
$$

The $f(1-f)$ factor cancels completely. This is the cancellation that squared error
cannot achieve, and it is the reason the vanishing gradient from section 4 does not
appear here. Finishing with $\partial z/\partial w_j = x_j$ and $\partial z/\partial b = 1$:

$$\boxed{\;\frac{\partial J}{\partial w_j} = \frac{1}{m}\sum_{i=1}^{m}\big(f_{w,b}(x^{(i)}) - y^{(i)}\big)x_j^{(i)}
\qquad
\frac{\partial J}{\partial b} = \frac{1}{m}\sum_{i=1}^{m}\big(f_{w,b}(x^{(i)}) - y^{(i)}\big)\;}$$

**These are character for character the formulas from lesson 01.** Only the meaning of
$f_{w,b}$ has changed, from $w \cdot x + b$ to $\sigma(w \cdot x + b)$. The code for
`compute_gradient` is identical in both lessons.

This is not a coincidence. Both models belong to the family of generalised linear models,
and every member of that family, fitted by maximum likelihood, produces a gradient of the
form (prediction minus label) times feature. Vectorised, with
$e = \sigma(Xw + b) - y$ of shape $(m,)$:

$$\nabla_w J = \frac{1}{m}X^\top e \qquad \frac{\partial J}{\partial b} = \frac{1}{m}\mathbf{1}^\top e$$

$X^\top$ has shape $(n, m)$ and $e$ has shape $(m,)$, so the product has shape $(n,)$,
matching $w$ as a gradient must.

In [ ]:
def numerical_gradient(X, y, w, b, eps=1e-6):
    identity = np.eye(len(w))
    dj_dw = np.array([
        (compute_cost(X, y, w + eps * identity[j], b)
         - compute_cost(X, y, w - eps * identity[j], b)) / (2 * eps)
        for j in range(len(w))
    ])
    dj_db = (compute_cost(X, y, w, b + eps) - compute_cost(X, y, w, b - eps)) / (2 * eps)
    return dj_dw, dj_db


w_test, b_test = np.array([0.7, -0.4]), 0.2
ana_w, ana_b = compute_gradient(X, y, w_test, b_test)
num_w, num_b = numerical_gradient(X, y, w_test, b_test)

print("analytic dJ/dw :", ana_w.round(9), "   dJ/db :", round(ana_b, 9))
print("numerical dJ/dw:", num_w.round(9), "   dJ/db :", round(num_b, 9))
print("\nmax absolute difference:", np.max(np.abs(np.append(ana_w - num_w, ana_b - num_b))))

## 7. Gradient descent

Unchanged from lesson 01, including the requirement that all parameters update
simultaneously from the gradient measured at the current values:

$$w_j := w_j - \alpha\frac{\partial J}{\partial w_j} \qquad b := b - \alpha\frac{\partial J}{\partial b}$$

In [ ]:
w_fit, b_fit, history = gradient_descent(X, y, np.zeros(2), 0.0, alpha=0.5, num_iters=3000)

print(f"learned w = {w_fit.round(4)},  b = {b_fit:.4f}")
print(f"cost at w = 0, b = 0     : {compute_cost(X, y, np.zeros(2), 0.0):.4f}  (this is log 2)")
print(f"cost after one update    : {history['cost'][0]:.4f}")
print(f"cost after {len(history['cost'])} updates : {history['cost'][-1]:.4f}")
print(f"training accuracy: {accuracy(y, (predict_proba(X, w_fit, b_fit) >= 0.5).astype(int)):.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["iter"], history["cost"])
axes[0].axhline(np.log(2), color="gray", ls=":", label="cost of always predicting 0.5")
axes[0].set_xlabel("iteration"); axes[0].set_ylabel("$J(w,b)$")
axes[0].legend(fontsize=8); axes[0].set_title("Learning curve")

plot_data(axes[1])
plot_boundary(axes[1], w_fit, b_fit, label="learned boundary")
axes[1].legend(fontsize=8); axes[1].set_title("The fitted decision boundary")
plt.tight_layout(); plt.show()

The dotted line marks $\log 2 \approx 0.693$, the cost of a model that outputs $0.5$ for
every example. That is the score to beat, and it is exactly what the starting values
$w = 0$ and $b = 0$ produce, since $\sigma(0) = 0.5$. Any useful model sits below it. The
plotted curve already starts below the line because its first recorded point comes after
one update has been applied.

### The model reports confidence, not just a class

The boundary is one contour of a smooth probability surface. Points far from the boundary
receive probabilities near $0$ or $1$, and points near it receive probabilities near
$0.5$. The magnitude of $w$ controls how quickly confidence changes as you cross.

In [ ]:
x1_grid = np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 200)
x2_grid = np.linspace(X[:, 1].min() - 1, X[:, 1].max() + 1, 200)
XX1, XX2 = np.meshgrid(x1_grid, x2_grid)
grid_points = np.column_stack([XX1.ravel(), XX2.ravel()])       # shape (40000, 2)
probabilities = predict_proba(grid_points, w_fit, b_fit).reshape(XX1.shape)

fig, ax = plt.subplots(figsize=(7, 5))
filled = ax.contourf(XX1, XX2, probabilities, levels=20, cmap="RdBu_r", alpha=0.55)
plt.colorbar(filled, ax=ax, label="$P(y = 1 \\mid x)$")
ax.contour(XX1, XX2, probabilities, levels=[0.25, 0.5, 0.75],
           colors="k", linewidths=[0.8, 2.0, 0.8], linestyles=["--", "-", "--"])
plot_data(ax)
ax.legend(fontsize=8); ax.set_title("Probability surface, with the 0.25, 0.5 and 0.75 contours")
plt.show()

## 8. Measuring a classifier

Accuracy is the fraction of correct predictions. It is the obvious metric and it is
misleading whenever the classes are imbalanced. If $99\%$ of transactions are legitimate,
a model that predicts "legitimate" every single time scores $99\%$ accuracy while
detecting no fraud at all.

The **confusion matrix** breaks the predictions into four counts and hides nothing:

|  | predicted 0 | predicted 1 |
|---|---|---|
| **actually 0** | true negative (TN) | false positive (FP) |
| **actually 1** | false negative (FN) | true positive (TP) |

From those four numbers:

$$\text{precision} = \frac{TP}{TP + FP} \qquad \text{recall} = \frac{TP}{TP + FN} \qquad F_1 = 2\,\frac{\text{precision} \cdot \text{recall}}{\text{precision} + \text{recall}}$$

- **Precision** answers: of everything I flagged as positive, how much really was? Low
  precision means false alarms.
- **Recall** answers: of everything that really was positive, how much did I catch? Low
  recall means misses.
- **$F_1$** is their harmonic mean, which stays low unless both are high.

Which one matters is a question about consequences, not about mathematics. A cancer
screen must not miss cases, so recall dominates. A spam filter must not bin a real
message, so precision dominates.

In [ ]:
y_pred = (predict_proba(X, w_fit, b_fit) >= 0.5).astype(int)
cm = confusion_matrix(y, y_pred)
precision, recall, f1 = precision_recall_f1(y, y_pred)

print("confusion matrix")
print(f"                predicted 0   predicted 1")
print(f"  actually 0    {cm[0,0]:>11}   {cm[0,1]:>11}")
print(f"  actually 1    {cm[1,0]:>11}   {cm[1,1]:>11}")
print(f"\naccuracy  = {accuracy(y, y_pred):.4f}")
print(f"precision = {precision:.4f}   of the {cm[:,1].sum()} flagged positive, {cm[1,1]} really were")
print(f"recall    = {recall:.4f}   of the {cm[1,:].sum()} actual positives, {cm[1,1]} were caught")
print(f"F1        = {f1:.4f}")

### The threshold is a dial you control

Nothing forces the threshold to be $0.5$. Because the model outputs a probability, you
can move the threshold after training and slide along the trade-off between precision and
recall, without refitting anything.

Lowering the threshold makes the model more willing to predict class 1, so it catches
more true positives and raises recall, while also raising false alarms and lowering
precision. Raising the threshold does the reverse.

In [ ]:
probs = predict_proba(X, w_fit, b_fit)
thresholds = np.linspace(0.02, 0.98, 97)
rows = [precision_recall_f1(y, (probs >= t).astype(int)) for t in thresholds]
prec_curve, rec_curve, f1_curve = np.array(rows).T
best = thresholds[f1_curve.argmax()]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(thresholds, prec_curve, label="precision")
axes[0].plot(thresholds, rec_curve, label="recall")
axes[0].plot(thresholds, f1_curve, "k--", label="$F_1$")
axes[0].axvline(0.5, color="gray", ls=":", label="default threshold")
axes[0].set_xlabel("threshold"); axes[0].set_ylabel("score"); axes[0].legend(fontsize=8)
axes[0].set_title("Moving the threshold trades one against the other")

axes[1].plot(rec_curve, prec_curve, lw=2)
axes[1].set_xlabel("recall"); axes[1].set_ylabel("precision")
axes[1].set_title("The precision against recall curve")
plt.tight_layout(); plt.show()

print(f"best F1 of {f1_curve.max():.4f} at threshold {best:.2f}")
for t in (0.1, 0.5, 0.9):
    p_t, r_t, f_t = precision_recall_f1(y, (probs >= t).astype(int))
    print(f"threshold {t:.1f}: precision {p_t:.3f}, recall {r_t:.3f}, F1 {f_t:.3f}")

## 9. Bending the decision boundary

The boundary $w \cdot x + b = 0$ is a straight line, so data that is not linearly
separable cannot be fitted well. The fix is the one from exercise 4 of lesson 01: invent
new features. The model stays linear in its parameters while the boundary becomes curved
in the original feature space.

For data where one class sits inside a circle, adding $x_1^2$ and $x_2^2$ is enough. The
boundary becomes

$$w_1x_1 + w_2x_2 + w_3x_1^2 + w_4x_2^2 + b = 0$$

which describes an ellipse. Squared features cover a much wider numeric range than the
originals, so scaling them is not optional here.

In [ ]:
m_c = 300
radius = rng.uniform(0, 3, m_c)
angle = rng.uniform(0, 2 * np.pi, m_c)
X_circ = np.column_stack([radius * np.cos(angle), radius * np.sin(angle)])
y_circ = (radius < 1.6).astype(float)              # class 1 is the inner disc
flip = rng.random(m_c) < 0.05                       # 5 percent label noise
y_circ[flip] = 1 - y_circ[flip]

X_poly = np.column_stack([X_circ, X_circ ** 2])     # shape (300, 4): x1, x2, x1^2, x2^2

linear_model = LogisticRegression(alpha=0.5, num_iters=4000, normalize=True).fit(X_circ, y_circ)
poly_model = LogisticRegression(alpha=0.5, num_iters=4000, normalize=True).fit(X_poly, y_circ)

majority_baseline = max(np.mean(y_circ), 1 - np.mean(y_circ))
noise_ceiling = 1 - np.mean(flip)
print(f"always predict the majority class : accuracy {majority_baseline:.3f}")
print(f"straight boundary, 2 features     : accuracy {linear_model.score(X_circ, y_circ):.3f}")
print(f"curved boundary, 4 features       : accuracy {poly_model.score(X_poly, y_circ):.3f}")
print(f"best any model could do here      : accuracy {noise_ceiling:.3f}  ({flip.sum()} labels were flipped)")

g1 = np.linspace(-3.2, 3.2, 250)
G1, G2 = np.meshgrid(g1, g1)
flat = np.column_stack([G1.ravel(), G2.ravel()])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, model, grid_features, train_accuracy, title in [
        (axes[0], linear_model, flat, linear_model.score(X_circ, y_circ), "linear features only"),
        (axes[1], poly_model, np.column_stack([flat, flat ** 2]),
         poly_model.score(X_poly, y_circ), "with squared features")]:
    # grid_features is the meshgrid laid out as (62500, n) for drawing the surface;
    # train_accuracy is measured on the 300 training rows, not on the grid
    surface = model.predict_proba(grid_features).reshape(G1.shape)
    ax.contourf(G1, G2, surface, levels=20, cmap="RdBu_r", alpha=0.5)
    ax.contour(G1, G2, surface, levels=[0.5], colors="k", linewidths=2)
    ax.scatter(X_circ[y_circ == 0, 0], X_circ[y_circ == 0, 1], c="tab:blue", s=14, alpha=0.8)
    ax.scatter(X_circ[y_circ == 1, 0], X_circ[y_circ == 1, 1], c="tab:red", s=14, alpha=0.8, marker="^")
    ax.set_xlabel("$x_1$"); ax.set_ylabel("$x_2$"); ax.set_aspect("equal")
    ax.set_title(f"{title}, accuracy {train_accuracy:.3f}")
plt.tight_layout(); plt.show()

With only $x_1$ and $x_2$ the best available straight line barely beats predicting the
majority class, because no line separates a disc from the ring around it. Adding the
squared features recovers the circular boundary and lands close to the noise ceiling,
which is the accuracy of a perfect model given that some labels were deliberately
flipped. Those flipped labels are exactly what a model should *not* fit.

The obvious next thought is to keep adding higher powers until everything is separable.
That leads directly to overfitting, which is lesson 03.

## 10. Checking against the module

`logistic_regression.py` packages all of this behind a class with the same shape as the
one in lesson 01.

In [ ]:
model = LogisticRegression(alpha=0.5, num_iters=3000).fit(X, y)

print("weights from the class      :", model.w.round(4), " bias:", round(model.b, 4))
print("weights from the loop above :", w_fit.round(4), " bias:", round(b_fit, 4))
print(f"\naccuracy {model.score(X, y):.4f}")

sample = np.array([[0.0, 0.0], [3.0, 2.0], [-3.0, -2.0]])
for point, prob in zip(sample, model.predict_proba(sample)):
    odds = prob / (1 - prob)
    print(f"x = {point},  P(y=1) = {prob:.4f},  odds = {odds:7.3f},  predicted class {int(prob >= 0.5)}")

## Exercises

1. **Read the weights.** Using the fitted model, confirm that increasing $x_1$ by one unit
   multiplies the odds by $e^{w_1}$. Compute the odds at two points that differ only in
   $x_1$ and check the ratio against $e^{w_1}$ directly.

2. **Make squared error fail on purpose.** Implement the squared error cost and its
   gradient for the logistic model, then fit the 1D data from section 4 starting from
   several different values of $w$. Find a starting point where it stalls far from the
   answer, and confirm cross-entropy reaches the same solution from every start.

3. **Class imbalance.** Build a dataset where only $2\%$ of the examples belong to class 1.
   Fit the model and report accuracy, precision and recall. Explain why accuracy looks
   excellent. Then find the threshold that maximises $F_1$ and report the three numbers
   again.

4. **Perfect separation.** Fit the model on two clusters that a straight line separates
   with a wide gap, and print the norm of $w$ every few hundred iterations. Explain why it
   keeps growing and never settles, and why the cost keeps decreasing towards zero
   without ever reaching it. This is the problem that regularisation solves in lesson 03.

5. **Mini-batch training.** Adapt the `sgd` function from the lesson 01 solutions to
   logistic regression. Since `compute_gradient` has the same signature in both lessons,
   check how little needs to change. Compare the learning curves for batch sizes 1, 16
   and the full dataset.

## What's next

Lesson 03, **regularisation**: the polynomial features in section 9 hint that a model
flexible enough to fit anything will fit the noise as well as the signal. We add a
penalty term to the cost that prefers small weights, and watch the bias and variance
trade off against each other.